## SRP041998

**paper:** [PMID: 25556892](https://onlinelibrary.wiley.com/doi/full/10.1111/1755-0998.12367?saml_referrer) - First insights into the giant panda (Ailuropoda melanoleuca) blood transcriptome: a resource for novel gene loci and immunogenetics, 2015

**date, curator:** 2026-04-01, Sara Carsanaro

**notes**
* [PMID: 30641486](https://www.aging-us.com/article/101747/text) is also relevant here

### annotation summary

In [20]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Blood,UBERON:0000178,blood,perfect match
2,blood,UBERON:0000178,blood,perfect match


In [21]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,2,UBERON:0000112,sexually immature stage,other
1,5,UBERON:0000112,sexually immature stage,other
2,6,UBERON:0000113,post-juvenile adult stage,other
3,12,UBERON:0000113,post-juvenile adult stage,other
4,18,UBERON:0000113,post-juvenile adult stage,other
5,19,UBERON:0000113,post-juvenile adult stage,other
6,21,UBERON:0000113,post-juvenile adult stage,other


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP041998"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████████| 7/7 [00:07<00:00,  1.09s/it]
curl: (22) The requested URL returned error: 500
>78 ERROR: >78 curl command failed ( Wed Apr  1 16:20:29 CEST 2026 ) with: 22>78
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi -d id=7423476%2C7423475%2C7423474%2C7423473%2C774443%2C774442%2C774431&WebEnv=MCID_69cd29ac70b64f01cd071533&dbfrom=sra&db=biosample&cmd=neighbor_history&linkname=sra_biosample&tool=edirect&edirect=16.2&edirect_os=Darwin&email=scarsana%40SORGE42778>78
HTTP/1.1 500 Internal Server Error
nquire -url https://eutils.ncbi.nlm.nih.

### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX5499802,SRP041998,Illumina HiSeq 2000,SRS4469017,,,UBERON:0000112,sexually immature stage,,Blood,2,,,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,,,,F02,SAMN11087557,2,year,,,,,,,,,01/04/2026,RNA-seq,,F02,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX5499804,SRP041998,Illumina HiSeq 2000,SRS4469019,,,UBERON:0000112,sexually immature stage,,Blood,5,,,other,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,,,,M05,SAMN11087559,5,year,,,,,,,,,01/04/2026,RNA-seq,,M05,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX542926,SRP041998,Illumina HiSeq 2000,SRS607025,,,UBERON:0000113,post-juvenile adult stage,,blood,6,,,other,F,,,9646,,,,,,XB02,SAMN02777953,6,year,,,,,,,,,01/04/2026,,,XB02,,,,,,TRANSCRIPTOMIC,PCR
3,SRX542925,SRP041998,Illumina HiSeq 2000,SRS607028,,,UBERON:0000113,post-juvenile adult stage,,blood,12,,,other,M,,,9646,,,,,,XB01,SAMN02777952,12,year,,,,,,,,,01/04/2026,,,XB01,,,,,,TRANSCRIPTOMIC,PCR
4,SRX5499803,SRP041998,Illumina HiSeq 2000,SRS4469018,,,UBERON:0000113,post-juvenile adult stage,,Blood,18,,,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,,,,F18,SAMN11087558,18,year,,,,,,,,,01/04/2026,RNA-seq,,F18,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX542914,SRP041998,Illumina HiSeq 2000,SRS607026,,,UBERON:0000113,post-juvenile adult stage,,blood,19,,,other,F,,,9646,,,,,,SL01,SAMN02777951,19,year,,,,,,,,,01/04/2026,,,SL01,,,,,,TRANSCRIPTOMIC,PCR
6,SRX5499805,SRP041998,Illumina HiSeq 2000,SRS4469020,,,UBERON:0000113,post-juvenile adult stage,,Blood,21,,,other,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,,,,M21,SAMN11087560,21,year,,,,,,,,,01/04/2026,RNA-seq,,M21,,,,,,TRANSCRIPTOMIC,cDNA


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Blood' 'blood']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0000178'
library.loc[:,'anatName'] = 'blood'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX5499802,SRP041998,Illumina HiSeq 2000,SRS4469017,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,Blood,2,perfect match,not documented,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,,,,F02,SAMN11087557,2,year,,,,,,,,,01/04/2026,RNA-seq,,F02,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX5499804,SRP041998,Illumina HiSeq 2000,SRS4469019,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,Blood,5,perfect match,not documented,other,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,,,,M05,SAMN11087559,5,year,,,,,,,,,01/04/2026,RNA-seq,,M05,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX542926,SRP041998,Illumina HiSeq 2000,SRS607025,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,6,perfect match,not documented,other,F,,,9646,,,,,,XB02,SAMN02777953,6,year,,,,,,,,,01/04/2026,,,XB02,,,,,,TRANSCRIPTOMIC,PCR
3,SRX542925,SRP041998,Illumina HiSeq 2000,SRS607028,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,12,perfect match,not documented,other,M,,,9646,,,,,,XB01,SAMN02777952,12,year,,,,,,,,,01/04/2026,,,XB01,,,,,,TRANSCRIPTOMIC,PCR
4,SRX5499803,SRP041998,Illumina HiSeq 2000,SRS4469018,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,18,perfect match,not documented,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,,,,F18,SAMN11087558,18,year,,,,,,,,,01/04/2026,RNA-seq,,F18,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX542914,SRP041998,Illumina HiSeq 2000,SRS607026,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,19,perfect match,not documented,other,F,,,9646,,,,,,SL01,SAMN02777951,19,year,,,,,,,,,01/04/2026,,,SL01,,,,,,TRANSCRIPTOMIC,PCR
6,SRX5499805,SRP041998,Illumina HiSeq 2000,SRS4469020,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,21,perfect match,not documented,other,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,,,,M21,SAMN11087560,21,year,,,,,,,,,01/04/2026,RNA-seq,,M21,,,,,,TRANSCRIPTOMIC,cDNA


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['12' '18' '19' '2' '21' '5' '6']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [8]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext Ultra RNA Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX5499802,SRP041998,Illumina HiSeq 2000,SRS4469017,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,Blood,2,perfect match,not documented,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,F02,SAMN11087557,2,year,,,,,,,,,01/04/2026,RNA-seq,,F02,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX5499804,SRP041998,Illumina HiSeq 2000,SRS4469019,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,Blood,5,perfect match,not documented,other,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,M05,SAMN11087559,5,year,,,,,,,,,01/04/2026,RNA-seq,,M05,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX542926,SRP041998,Illumina HiSeq 2000,SRS607025,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,6,perfect match,not documented,other,F,,,9646,,,polyA,,,XB02,SAMN02777953,6,year,,,,,,,,,01/04/2026,,,XB02,,,,,,TRANSCRIPTOMIC,PCR
3,SRX542925,SRP041998,Illumina HiSeq 2000,SRS607028,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,12,perfect match,not documented,other,M,,,9646,,,polyA,,,XB01,SAMN02777952,12,year,,,,,,,,,01/04/2026,,,XB01,,,,,,TRANSCRIPTOMIC,PCR
4,SRX5499803,SRP041998,Illumina HiSeq 2000,SRS4469018,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,18,perfect match,not documented,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,F18,SAMN11087558,18,year,,,,,,,,,01/04/2026,RNA-seq,,F18,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX542914,SRP041998,Illumina HiSeq 2000,SRS607026,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,19,perfect match,not documented,other,F,,,9646,,,polyA,,,SL01,SAMN02777951,19,year,,,,,,,,,01/04/2026,,,SL01,,,,,,TRANSCRIPTOMIC,PCR
6,SRX5499805,SRP041998,Illumina HiSeq 2000,SRS4469020,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,21,perfect match,not documented,other,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,M21,SAMN11087560,21,year,,,,,,,,,01/04/2026,RNA-seq,,M21,,,,,,TRANSCRIPTOMIC,cDNA


#### globin, replicates

In [9]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [10]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-01'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX5499802,SRP041998,Illumina HiSeq 2000,SRS4469017,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,Blood,2,perfect match,not documented,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,F02,SAMN11087557,2,year,,,,,,,,SAC,2026-04-01,RNA-seq,,F02,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX5499804,SRP041998,Illumina HiSeq 2000,SRS4469019,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,Blood,5,perfect match,not documented,other,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,M05,SAMN11087559,5,year,,,,,,,,SAC,2026-04-01,RNA-seq,,M05,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX542926,SRP041998,Illumina HiSeq 2000,SRS607025,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,6,perfect match,not documented,other,F,,,9646,,,polyA,,,XB02,SAMN02777953,6,year,,,,,,,,SAC,2026-04-01,,,XB02,,,,,,TRANSCRIPTOMIC,PCR
3,SRX542925,SRP041998,Illumina HiSeq 2000,SRS607028,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,12,perfect match,not documented,other,M,,,9646,,,polyA,,,XB01,SAMN02777952,12,year,,,,,,,,SAC,2026-04-01,,,XB01,,,,,,TRANSCRIPTOMIC,PCR
4,SRX5499803,SRP041998,Illumina HiSeq 2000,SRS4469018,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,18,perfect match,not documented,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,F18,SAMN11087558,18,year,,,,,,,,SAC,2026-04-01,RNA-seq,,F18,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX542914,SRP041998,Illumina HiSeq 2000,SRS607026,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,19,perfect match,not documented,other,F,,,9646,,,polyA,,,SL01,SAMN02777951,19,year,,,,,,,,SAC,2026-04-01,,,SL01,,,,,,TRANSCRIPTOMIC,PCR
6,SRX5499805,SRP041998,Illumina HiSeq 2000,SRS4469020,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,21,perfect match,not documented,other,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,M21,SAMN11087560,21,year,,,,,,,,SAC,2026-04-01,RNA-seq,,M21,,,,,,TRANSCRIPTOMIC,cDNA


#### comments

In [11]:
library.loc[:,'comment'] = 'PMID: 30641486, PMID: 25556892'

#### save complete file with correct columns

In [12]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX5499802,SRP041998,Illumina HiSeq 2000,SRS4469017,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,Blood,2,perfect match,not documented,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,F02,SAMN11087557,2,year,,,,,"PMID: 30641486, PMID: 25556892",,,SAC,2026-04-01
1,SRX5499804,SRP041998,Illumina HiSeq 2000,SRS4469019,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,Blood,5,perfect match,not documented,other,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,M05,SAMN11087559,5,year,,,,,"PMID: 30641486, PMID: 25556892",,,SAC,2026-04-01
2,SRX542926,SRP041998,Illumina HiSeq 2000,SRS607025,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,6,perfect match,not documented,other,F,,,9646,,,polyA,,,XB02,SAMN02777953,6,year,,,,,"PMID: 30641486, PMID: 25556892",,,SAC,2026-04-01
3,SRX542925,SRP041998,Illumina HiSeq 2000,SRS607028,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,12,perfect match,not documented,other,M,,,9646,,,polyA,,,XB01,SAMN02777952,12,year,,,,,"PMID: 30641486, PMID: 25556892",,,SAC,2026-04-01
4,SRX5499803,SRP041998,Illumina HiSeq 2000,SRS4469018,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,18,perfect match,not documented,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,F18,SAMN11087558,18,year,,,,,"PMID: 30641486, PMID: 25556892",,,SAC,2026-04-01
5,SRX542914,SRP041998,Illumina HiSeq 2000,SRS607026,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,19,perfect match,not documented,other,F,,,9646,,,polyA,,,SL01,SAMN02777951,19,year,,,,,"PMID: 30641486, PMID: 25556892",,,SAC,2026-04-01
6,SRX5499805,SRP041998,Illumina HiSeq 2000,SRS4469020,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,21,perfect match,not documented,other,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,M21,SAMN11087560,21,year,,,,,"PMID: 30641486, PMID: 25556892",,,SAC,2026-04-01


### experiment annotations

In [13]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP041998,The giant panda blood Transcriptome,Assemble and characterize the blood transcriptome of the giant panda,SRA,,,,,,,PRJNA247712,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [14]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

7

In [15]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP041998,The giant panda blood Transcriptome,Assemble and characterize the blood transcriptome of the giant panda,SRA,total,Bgee 1K,7,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA247712,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [16]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '30641486'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6339791/'
experiment.loc[:,'DOI'] = '10.18632/aging.101747'
experiment.loc[:,'xrefs'] = '25556892[PMID]'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP041998,The giant panda blood Transcriptome,Assemble and characterize the blood transcriptome of the giant panda,SRA,total,Bgee 1K,7,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA247712,30641486,https://pmc.ncbi.nlm.nih.gov/articles/PMC6339791/,10.18632/aging.101747,25556892[PMID],


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [17]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [18]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [19]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [22]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [23]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
58812,SRX15138584,SRP373446,Illumina NovaSeq 6000,SRS12881931,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,whole blood,22,perfect match,not documented,other,M,,,1945556,Ovation SoLo RNA-Seq library preparation kit,full_length,ribo-minus,,,37891_Null,SAMN28064227,22,month,,,,,PMID: 38917861,,,SAC,2026-04-01
58813,SRX15138583,SRP373446,Illumina NovaSeq 6000,SRS12881932,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,whole blood,23,perfect match,not documented,other,M,,,1945556,Ovation SoLo RNA-Seq library preparation kit,full_length,ribo-minus,,,37859_Null,SAMN28064226,23,month,,,,,PMID: 38917861,,,SAC,2026-04-01
58814,SRX5499802,SRP041998,Illumina HiSeq 2000,SRS4469017,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,Blood,2,perfect match,not documented,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,F02,SAMN11087557,2,year,,,,,"PMID: 30641486, PMID: 25556892",,,SAC,2026-04-01
58815,SRX5499804,SRP041998,Illumina HiSeq 2000,SRS4469019,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,Blood,5,perfect match,not documented,other,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,M05,SAMN11087559,5,year,,,,,"PMID: 30641486, PMID: 25556892",,,SAC,2026-04-01
58816,SRX542926,SRP041998,Illumina HiSeq 2000,SRS607025,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,6,perfect match,not documented,other,F,,,9646,,,polyA,,,XB02,SAMN02777953,6,year,,,,,"PMID: 30641486, PMID: 25556892",,,SAC,2026-04-01
58817,SRX542925,SRP041998,Illumina HiSeq 2000,SRS607028,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,blood,12,perfect match,not documented,other,M,,,9646,,,polyA,,,XB01,SAMN02777952,12,year,,,,,"PMID: 30641486, PMID: 25556892",,,SAC,2026-04-01
58818,SRX5499803,SRP041998,Illumina HiSeq 2000,SRS4469018,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,18,perfect match,not documented,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,F18,SAMN11087558,18,year,,,,,"PMID: 30641486, PMID: 25556892",,,SAC,2026-04-01


In [24]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1141,SRP228410,Genomic and RNA-Seq raw sequence reads from th...,Transcript variation has important implication...,SRA,partial,Bgee 1K,6,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA587121,27413049,https://pmc.ncbi.nlm.nih.gov/articles/PMC5026258/,10.1093/molbev/msw139,,"removed DNA libraries, note that animal died f..."
1142,SRP373446,Allometry of innate immune gene transcription ...,Allometry of innate immune gene transcription ...,SRA,partial,Bgee 1K,16,Ovation SoLo RNA-Seq library preparation kit,full_length,,PRJNA834816,38917861,https://pmc.ncbi.nlm.nih.gov/articles/PMC11285...,10.1098/rspb.2024.0535,,removed bacterial lipopolysaccharide samples
1143,SRP041998,The giant panda blood Transcriptome,Assemble and characterize the blood transcript...,SRA,total,Bgee 1K,7,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA247712,30641486,https://pmc.ncbi.nlm.nih.gov/articles/PMC6339791/,10.18632/aging.101747,25556892[PMID],


### add annotations to git

In [25]:
! git pull

Already up to date.


In [26]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [27]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../1_reject_experiment/
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop b545f68] adding annotated bulk experiment SRP041998
 2 files changed, 8 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.02 KiB | 2.02 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   9892f3a..b545f68  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push